In [1]:
import os
from openai import OpenAI
import gradio as gr
import sqlite3
import json
import time




In [3]:
openai=OpenAI()


system_prompt="""
You are a helpful assistant for our flight booking website FlyHigh.
you have access to a database of flights and their availability.
you have access to tool to figure out the distace based on the closest airport that you have access to 
for example  if user want to travel from mumbai to delhi you will use the tool to figure out the distance between the two cities and then use the tool to figure out the price or even you can sugges the closes sure shour price 

"""




In [6]:
DB="flights.db"
with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute("CREATE TABLE IF NOT EXISTS flights (id INTEGER PRIMARY KEY AUTOINCREMENT, city TEXT, price INTEGER)")
    cursor.execute("INSERT INTO flights (city, price) VALUES ('Mumbai', 1000)")
    cursor.execute("INSERT INTO flights (city, price) VALUES ('Delhi', 2000)")
    cursor.execute("INSERT INTO flights (city, price) VALUES ('Chennai', 3000)")
    cursor.execute("INSERT INTO flights (city, price) VALUES ('Kolkata', 4000)")
    cursor.execute("INSERT INTO flights (city, price) VALUES ('Bangalore', 5000)")
    conn.commit()
    


In [45]:
def get_price(city):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT price FROM flights WHERE city = ?", (city,))
        price = cursor.fetchone()
        return f"{price[0]}"


In [46]:
get_price("Mumbai")

'1000'

In [47]:
def set_price(distance,city):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("Insert into flights (city,price) values (?,?)",(city,distance*.10))
        conn.commit()
    return f"Price set to {distance*.10} for {city}"





In [ ]:
def handle_tool_calls(message):
    rs=[]
    tool_calls = message.tool_calls
    for tool_call in tool_calls:
        if tool_call.function.name == "set_price":
            dist = tool_call.function.arguments.get("distance_btw_two_city")
            city= tool.calll.function.arguments.get("destination_city")
            price = set_price(dist,city)
            res.append({"role":"tool","content":price,"tool_call_id":tool_call.id})
        if tool_call.function.name == "get_price":
            city=tool.calll.function.arguments.get("city")
            price=get_price(city)
            res.append({"role":"tool","content":price,"tool_call_id":tool_call.id})
    return res


In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "set_price",
            "description": "set the price based on the distance between the two cities",
            "parameters": {
                "type": "object",
                "properties": {
                    "distance_btw_two_city": {
                        "type": "integer",
                        "description": "the distance between the two cities egfrom delhi to mumbai will be 1000"
                    },
                    "destination_city":{
                        "type":"string",
                        "description":"the destnation city user wanted to go eg mumbai "
                    }

                },
                
                "required": ["distance_btw_two_city","destination_city"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_price",
            "description": "get the price of the flight",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "the city to get the price of the flight"
                    }
                },
                "required": ["city"],
                "additionalProperties": False
            }
        }
    }
]

In [56]:
MODEL = "gpt-4.1-mini"
def chat (message,history):

    history = [{"role":h["role"], "content":h["content"]} for h in history]
    msgs = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    res=openai.chat.completions.create(
        model=MODEL,
        messages=msgs,
        tools=tools,
            
    )
    while res.choices[0].finish_reason=="tool_calls":

        message = res.choices[0].message
        responses = handle_tool_calls(message)
        msgs.append(message)
        msgs.extend(responses)
        res = openai.chat.completions.create(model=MODEL, messages=msgs, tools=tools)

    return res.choices[0].message.content


In [57]:
gr.ChatInterface(chat,title="Flight Booking",description="Book your flight with FlyHigh",type="messages").launch()

* Running on local URL:  http://127.0.0.1:7875
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "/home/shekhar/Personal/llm_engineering/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/shekhar/Personal/llm_engineering/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/shekhar/Personal/llm_engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2116, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/shekhar/Personal/llm_engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1621, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/shekhar/Personal/llm_engineering/.venv/lib/python3.12/site-pac